# Phase 26: Baseline vs Fine-tuned Evaluation

Run the **base Qwen3.5-4B** (no QLoRA adapter) on the 254 holdout to get pre-finetuning metrics,
then compare against the fine-tuned results.

**Requirements:**
- Colab with GPU (T4 minimum, H100/A100 preferred for speed)
- Upload `val.jsonl` from `data/splits/recovered-balanced/val.jsonl`

**Output:** `baseline-eval-results.json` — download and add comparison table to report

**Time estimate:** ~10-20 min on H100, ~30-60 min on T4

## Step 1: Install dependencies

In [ ]:
%%capture
%pip install -q transformers>=4.46.0 accelerate>=1.0.0 torch

## Step 2: Clone repo + upload val split

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wikiepeidia/Internship-project.git"
REPO_ROOT = Path("/content/vnphish-repo")
VAL_SPLIT = Path("/content/val.jsonl")

if not REPO_ROOT.exists():
    print("Cloning repo...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
else:
    print("Repo exists, pulling latest...")
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull"], check=False)

if not VAL_SPLIT.exists():
    raise FileNotFoundError(
        "Upload val.jsonl to /content/ first!\n"
        "Local path: data/splits/recovered-balanced/val.jsonl\n"
        "Use Colab Files panel -> Upload"
    )
print(f"Val split ready: {VAL_SPLIT} ({VAL_SPLIT.stat().st_size // 1024} KB)")

## Step 3: Download base Qwen3.5-4B (no adapter)

In [ ]:
import os, torch, json
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL_ID = "Qwen/Qwen3.5-4B"
MODEL_DIR = Path("/content/base-model")
MODEL_DIR.mkdir(exist_ok=True)

print(f"Downloading {BASE_MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, cache_dir=str(MODEL_DIR))
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto", cache_dir=str(MODEL_DIR)
)
model.eval()
print(f"Model loaded: {BASE_MODEL_ID}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## Step 4: Run baseline evaluation (NO adapter — raw base model)

Uses the model's native chat template for a fair zero-shot baseline.

In [ ]:
import json, sys
from collections import defaultdict
from pathlib import Path

LABELS = ["bank_impersonation", "zalo_social_engineering", "task_scam", "benign"]

SYSTEM_PROMPT = """You are a Vietnamese financial phishing detector.
Classify the following message into exactly one label:
- bank_impersonation
- zalo_social_engineering
- task_scam
- benign

Reply with ONLY the label, nothing else."""

def classify_base(text: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=32, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
    return next((lbl for lbl in LABELS if lbl in raw), "unknown")

val_rows = [json.loads(l) for l in open(VAL_SPLIT, encoding="utf-8")]
print(f"Evaluating {len(val_rows)} holdout examples with BASE model (no adapter)...")

results = []
for i, row in enumerate(val_rows):
    pred = classify_base(row["text"])
    results.append({"true": row["label"], "pred": pred})
    if (i + 1) % 25 == 0:
        correct_so_far = sum(1 for r in results if r["true"] == r["pred"])
        print(f"  {i+1}/{len(val_rows)} — running accuracy: {correct_so_far/(i+1):.3f}")

print("\nBaseline inference complete.")

## Step 5: Compute metrics + comparison table

In [ ]:
from collections import defaultdict

# --- Baseline metrics ---
tp = defaultdict(int); fp = defaultdict(int); fn = defaultdict(int)
for r in results:
    if r["true"] == r["pred"]: tp[r["true"]] += 1
    else: fp[r["pred"]] += 1; fn[r["true"]] += 1

baseline_metrics = {}
print("\n" + "="*70)
print("BASELINE (Qwen3.5-4B, no fine-tuning, zero-shot)")
print("="*70)
for lbl in LABELS:
    support = sum(1 for r in results if r["true"] == lbl)
    prec = tp[lbl] / (tp[lbl] + fp[lbl]) if (tp[lbl] + fp[lbl]) > 0 else 0.0
    rec  = tp[lbl] / (tp[lbl] + fn[lbl]) if (tp[lbl] + fn[lbl]) > 0 else 0.0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0.0
    baseline_metrics[lbl] = {"precision": prec, "recall": rec, "f1": f1, "support": support}
    print(f"  {lbl:35s}  prec={prec:.4f}  recall={rec:.4f}  f1={f1:.4f}  n={support}")

correct = sum(1 for r in results if r["true"] == r["pred"])
baseline_macro_f1 = sum(m["f1"] for m in baseline_metrics.values()) / len(LABELS)
baseline_accuracy = correct / len(results)
print(f"\n  Macro F1: {baseline_macro_f1:.4f}")
print(f"  Accuracy: {baseline_accuracy:.4f} ({correct}/{len(results)})")

# --- Fine-tuned metrics (known from report) ---
finetuned = {
    "bank_impersonation":     {"precision": 0.836, "recall": 1.000, "f1": 0.911, "support": 56},
    "zalo_social_engineering": {"precision": 1.000, "recall": 0.960, "f1": 0.980, "support": 75},
    "task_scam":              {"precision": 1.000, "recall": 0.871, "f1": 0.931, "support": 62},
    "benign":                 {"precision": 1.000, "recall": 1.000, "f1": 1.000, "support": 61},
}
finetuned_macro_f1 = 0.9553

# --- Comparison ---
print("\n" + "="*70)
print("COMPARISON: Baseline vs Fine-tuned (QLoRA)")
print("="*70)
print(f"{'Class':35s} {'Base F1':>8s}  {'Fine F1':>8s}  {'Delta':>8s}")
print("-"*70)
for lbl in LABELS:
    bf1 = baseline_metrics[lbl]["f1"]
    ff1 = finetuned[lbl]["f1"]
    delta = ff1 - bf1
    print(f"  {lbl:35s} {bf1:>7.4f}   {ff1:>7.4f}   {delta:>+7.4f}")
print("-"*70)
delta_macro = finetuned_macro_f1 - baseline_macro_f1
print(f"  {'Macro F1':35s} {baseline_macro_f1:>7.4f}   {finetuned_macro_f1:>7.4f}   {delta_macro:>+7.4f}")
print()

## Step 6: Save results JSON (download this)

In [ ]:
output = {
    "base_model": BASE_MODEL_ID,
    "holdout_size": len(results),
    "baseline": {
        "macro_f1": round(baseline_macro_f1, 4),
        "accuracy": round(baseline_accuracy, 4),
        "per_class": {k: {kk: round(vv, 4) for kk, vv in v.items()} for k, v in baseline_metrics.items()},
    },
    "finetuned": {
        "macro_f1": finetuned_macro_f1,
        "per_class": finetuned,
    },
    "delta_macro_f1": round(finetuned_macro_f1 - baseline_macro_f1, 4),
    "raw_predictions": results,
}

OUT_PATH = Path("/content/baseline-eval-results.json")
OUT_PATH.write_text(json.dumps(output, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Results saved to {OUT_PATH}")
print(f"\nDownload from Colab Files panel -> /content/baseline-eval-results.json")
print(f"Then bring the numbers back to the report (ch05 comparison table + slides).")

## Instructions

### Before running:
1. Open this notebook in Google Colab
2. Set runtime to GPU (T4/A100/H100)
3. Upload `val.jsonl` from `data/splits/recovered-balanced/val.jsonl` to `/content/`

### Run all cells in order (Step 1 through Step 6)

### After running:
1. Download `/content/baseline-eval-results.json` from the Files panel
2. The comparison table shows baseline vs fine-tuned F1 per class
3. Add these numbers to:
   - **Report ch05**: new table comparing base vs QLoRA metrics
   - **Slides**: evaluation slide update
4. The key narrative: base Qwen3.5-4B without fine-tuning scores X macro F1;
   after QLoRA adaptation it reaches 0.9553 — showing fine-tuning is necessary